### Business Understanding

1. Problem Statement:
    
    Perusahaan Fintech menghadapi resiko gagal bayar karna tidak semua nasabah mampu melunasi pinjaman. Ada dua jenis kesalahan yang perlu diminimalkan yaitu: Menyetujui nasabah yang gagal bayar yang menyebabkan kerugian finansial secara langsung dan Menolak nasabah yang sebenarnya mampu bayar yang menyebabkan hilangnya potensi pendapatan.
    Maka perusahaan perlu membangun model prediksi probabilitas gagal bayar agar keputusan pemberian kredit lebih akurat.

2. Objective:
    - Membangun model ML yang menghasilkan skor probabilitas gagal bayar untuk setiap pengajuan pinjaman.
    - Mengidentifikasi fitur-fitur paling berpengaruh untuk mendukung interpretabilitas kepada stakeholder non-teknis

3. Target Variable:
    
    Loan Status
    - Label  0 (Good)
    - Label  1 (Bad)

4. Metrics:
    - Recall (Prioritas Utama)
    - ROC-AUC
    - F1-Score
    - Precision

### Data Understanding

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('loan_data_2007_2014.csv')
print('Dataset berhasil dimuat!')
print('\nUkuran dataset:', df.shape)
print('\n5 Baris teratas:')
df.head()

In [ ]:
print('Basic info:')
df.info()

In [ ]:
# Check missing values
missing_count = df.isnull().sum()
missing_pct = missing_count / len(df) * 100
missing = pd.DataFrame({
    'Columns': missing_count.index,
    'Missing_Count': missing_count.values,
    'Missing_Pct': missing_pct.values
})
missing = missing[missing['Missing_Count'] > 0].sort_values('Missing_Pct', ascending=False)
print('Jumlah kolom dengan missing values:', len(missing))
print(missing.head(25))

In [ ]:
print('Jumlah data duplikat:', df.duplicated().sum())
print('\nRingkasan statistik numerik:')
df.describe()

In [ ]:
print('Distribusi target (loan_status):')
print(df['loan_status'].value_counts())

### Data Preparation

In [ ]:
# Mapping Target
bad_loan = [
    'Default',
    'Charged Off',
    'In Grace Period',
    'Late (16-30 days)',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off'
]
good_loan = [
    'Current',
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid'
]
final_loan = bad_loan + good_loan
 
df = df[df['loan_status'].isin(final_loan)].copy()
df['loan_status'] = df['loan_status'].isin(bad_loan).astype(int)
 
print('\nDistribusi target (loan_status):')
print(df['loan_status'].value_counts())

In [ ]:
leakage_cols = [
    'Unnamed: 0', 'id', 'member_id', 'url','desc', 'title', 'zip_code', 'emp_title',
    'policy_code', 'pymnt_plan','issue_d','funded_amnt', 'funded_amnt_inv',
    'total_pymnt', 'total_pymnt_inv','total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee','last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 
    'last_credit_pull_d','out_prncp', 'out_prncp_inv'
   ]

df.drop(columns=leakage_cols, errors='ignore', inplace=True)

print('Kolom leakage/tidak berguna berhasil dihapus!')
print('\nUkuran dataset terbaru:', df.shape)

In [ ]:
def convert_types(df):
    df = df.copy()

    df['term'] = df['term'].str.replace('months', '').str.strip().astype(float)
    df['emp_length'] = (
        df['emp_length']
        .replace({'< 1 year': '0', '10+ years': '10', 'n/a': None})
        .astype(str)
        .str.extract(r'(\d+)')[0]
        .astype(float))
    df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%y', errors='coerce')

    return df
df = convert_types(df)
print('\nKonversi tipe data selesai!')

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # Rasio cicilan terhadap pendapatan bulanan
    df['installment_income_ratio'] = df['installment'] / (df['annual_inc'] / 12)
    # Rasio revolving balance terhadap total limit
    df['revol_bal_ratio'] = df['revol_bal'] / df['total_rev_hi_lim']

    ref_date = pd.Timestamp('2014-12-31')
    df['credit_history_age'] = (ref_date - df['earliest_cr_line']).dt.days / 365.25
    # Batasi usia minimal 0
    df['credit_history_age'] = df['credit_history_age'].clip(lower=0)
    # Flag jika pernah ada delinquency (berdasarkan mths_since_last_delinq)
    df['has_delinq'] = df['mths_since_last_delinq'].notnull().astype(int)

    return df
df = feature_engineering(df)
print('Feature engineering selesai!')

In [ ]:
def handle_missing_values(df, threshold=70):
    df = df.copy()

    missing_pct = df.isnull().mean() * 100
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

    df.drop(columns=cols_to_drop, inplace=True)

    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(
                df[col].median()
            )
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(
                df[col].mode()[0]
            )

    return df
df = handle_missing_values(df)
print('Handle missing values selesai!')

In [ ]:
# EDA — Visualisasi
# Distribusi fitur numerik
num_features = ['loan_amnt', 'annual_inc', 'dti']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, num_features):
    df[col].hist(bins=50, ax=ax)
    ax.set_title(f'Distribusi {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur kategorikal
cat_features = ['grade', 'term']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
df['grade'].value_counts().reindex(grade_order).plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribusi grade')
axes[0].tick_params(axis='x', rotation=0)
 
df['term'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Distribusi term')
axes[1].tick_params(axis='x', rotation=0)
 
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot fitur vs loan_status
fig, axes = plt.subplots(1, len(num_features), figsize=(5 * len(num_features), 4))
for ax, col in zip(axes, num_features):
    sns.boxplot(x='loan_status', y=col, data=df, ax=ax)
    ax.set_title(f'{col} vs Loan Status')
    ax.set_xlabel('Loan Status (0=Good | 1=Bad)')
plt.tight_layout()
plt.show()

In [ ]:
# Default rate per kategori
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, cat_features):
    default_rate = (
        df.groupby(col)['loan_status']
        .mean()
        .sort_values(ascending=False)
    )
    default_rate.plot(kind='bar', ax=ax)
    ax.set_title(f'Default Rate per {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Default Rate')
    ax.tick_params(axis='x', rotation=0)
    for i, v in enumerate(default_rate):
        ax.text(i, v + 0.005, f'{v:.1%}', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
num_cols = df.select_dtypes(include=[np.number])
corr     = num_cols.corr()['loan_status'].abs().sort_values(ascending=False).head(15)
plt.figure(figsize=(12, 8))
sns.heatmap(
    num_cols[corr.index].corr(),
    cmap='coolwarm', annot=True, fmt='.2f', linewidths=0.5
)
plt.title('Top 15 Korelasi Heatmap')
plt.show()

In [ ]:
# Encoding
# ordinal encoding grade
grade_map = {'A': 1,'B': 2,'C': 3,'D': 4,'E': 5,'F': 6,'G': 7}

df['grade'] = df['grade'].map(grade_map)
# sub grade
sub_grades = [
    f'{g}{i}'
    for g in 'ABCDEFG'
    for i in range(1, 6)
]

sub_grade_map = {
    sg: i+1
    for i, sg in enumerate(sub_grades)
}

df['sub_grade'] = (
    df['sub_grade']
    .map(sub_grade_map)
)

# one hot encoding
cat_cols = [
    'home_ownership',
    'verification_status',
    'purpose'
]

df = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True
)
print('\nEncoding selesai!')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Split feature
X = df.drop(columns=['loan_status'])
y = df['loan_status']

X = X.select_dtypes(include=np.number)

print('\nJumlah fitur final:', X.shape[1])

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('\nTrain shape:', X_train.shape)
print('Test shape :', X_test.shape)

# Hitung scale_pos_weight dari data training asli
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'\nScale Pos Weight: {scale_pos_weight:.2f}')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('\nTrain shape:', X_train.shape)
print('Test shape :', X_test.shape)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(
    random_state=42,
    sampling_strategy=0.3
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled,
    y_train
)

print('\nDistribusi target sebelum SMOTE:')
print(y_train.value_counts(normalize=True))

print('\nDistribusi target sesudah SMOTE:')
print(y_train_resampled.value_counts(normalize=True))

print('\nShape setelah SMOTE:')
print('X_train_resampled:', X_train_resampled.shape)
print('y_train_resampled:', y_train_resampled.shape)
print('X_test_scaled    :', X_test_scaled.shape)

### Modeling

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

best_models = {}

lr = LogisticRegression(
    random_state=42,
    class_weight='balanced',
    max_iter=1000
)

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10]
}

grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train_resampled, y_train_resampled)

print(f"Best Recall (CV): {grid_lr.best_score_:.4f}")
print(f"Best Params     : {grid_lr.best_params_}")

best_models['Logistic Regression'] = grid_lr.best_estimator_

In [ ]:
xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1
)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# XGBoost menggunakan data training asli
grid_xgb.fit(X_train, y_train)

print(f"Best Recall (CV): {grid_xgb.best_score_:.4f}")
print(f"Best Params     : {grid_xgb.best_params_}")

best_models['XGBoost'] = grid_xgb.best_estimator_

In [ ]:
lgb = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_lgb = GridSearchCV(
    estimator=lgb,
    param_grid=param_grid_lgb,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# LightGBM menggunakan data training asli
grid_lgb.fit(X_train, y_train)

print(f"Best Recall (CV): {grid_lgb.best_score_:.4f}")
print(f"Best Params     : {grid_lgb.best_params_}")

best_models['LightGBM'] = grid_lgb.best_estimator_

### Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
)

results = []
predictions = {}

for model_name, model in best_models.items():
    print(f"Evaluasi: {model_name}")

    # Logistic Regression menggunakan data test yang sudah di-scale
    if model_name == 'Logistic Regression':
        X_eval = X_test_scaled
    else:
        X_eval = X_test

    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    predictions[model_name] = {
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"Recall    : {recall:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print()
    print(classification_report(
        y_test,
        y_pred,
        target_names=['Good (0)', 'Bad (1)']
    ))

    results.append({
        'Model': model_name,
        'Recall': recall,
        'Precision': precision,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })

In [ ]:
# Ringkasan hasil
results = (
    pd.DataFrame(results)
    .sort_values(by='Recall', ascending=False)
    .reset_index(drop=True)
)
print('Perbandingan Model:')
print(results.round(4))

In [ ]:
plt.figure(figsize=(8, 6))

for _, row in results.iterrows():
    model_name = row['Model']
    y_proba = predictions[model_name]['y_proba']

    fpr, tpr, _ = roc_curve(y_test, y_proba)

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {row['ROC-AUC']:.3f})"
    )

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Perbandingan Model')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
for model_name, pred_dict in predictions.items():
    # Hitung confusion matrix
    cm = confusion_matrix(y_test, pred_dict['y_pred'])
    
    # Plot heatmap
    plt.figure(figsize=(8, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Good (0)', 'Bad (1)'],
        yticklabels=['Good (0)', 'Bad (1)']
    )

    plt.xlabel('Predicted Label')
    plt.ylabel('Actual Label')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

In [ ]:
models_for_importance = {
    'Logistic Regression': best_models['Logistic Regression'],
    'XGBoost': best_models['XGBoost'],
    'LightGBM': best_models['LightGBM']
}

for model_name, model in models_for_importance.items():
    # Ambil importance sesuai tipe model
    if model_name == 'Logistic Regression':
        importance_values = np.abs(model.coef_[0])
    else:
        importance_values = model.feature_importances_

    # Buat DataFrame importance
    importance_df = (
        pd.DataFrame({
            'feature': X_train.columns,
            'importance': importance_values
        })
        .sort_values('importance', ascending=False)
        .head(15)
        .sort_values('importance', ascending=True)
    )

    # Plot
    plt.figure(figsize=(8, 6))
    plt.barh(
        importance_df['feature'],
        importance_df['importance']
    )
    plt.title(f'{model_name} - Top 15 Feature Importance')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()